In [4]:
!pip install corus
!pip install datasets
!pip install evaluate
!pip install seqeval

In [5]:
import pathlib as pl
import requests
import zipfile
from corus import load_ne5, load_lenta
import re
from datasets import Dataset, Sequence, ClassLabel
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer,
    DataCollatorForTokenClassification
)
import evaluate
import numpy as np
import tabulate
import torch

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

cuda


In [7]:
url = "http://www.labinform.ru/pub/named_entities/collection5.zip"

path = "Collection5.zip"
zip_path = pl.Path.cwd()/path

if not pl.Path.exists(zip_path):
    response = requests.get(url, stream=True)
    if response.status_code == 200:
        with open(path, "wb") as file:
            for chunk in response.iter_content(chunk_size=8192):
                file.write(chunk)
        print("Файл успешно загружен.")
    else:
        print("Ошибка при загрузке файла")
else:
    print("Файл уже существует")

Файл успешно загружен.


In [8]:
extract_dir = pl.Path.cwd()
if pl.Path.exists(zip_path):
    with zipfile.ZipFile(path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    if not pl.Path.exists(extract_dir):
        pl.Path.mkdir(extract_dir, exist_ok=True)

    print("Архив распакован")
else:
    print("Файл не существует.")

Архив распакован


In [9]:
dataset = load_ne5("Collection5")

In [10]:
next(dataset)

Ne5Markup(
    id='23_11_12e',
    text='И.о. главы Подмосковья назначил зама по политическим вопросам\r\nМОСКВА, 21 ноя — РИА Новости. Исполняющий обязанности губернатора Московской области Андрей Воробьев назначил своим заместителем по политическим вопросам и вопросам взаимодействия с общественными организациями Юрия Олейникова, который ранее был заместителем полномочного представителя президента РФ в Северо-Кавказском федеральном округе, сообщила РИА Новости советник Воробьева по информационной политике Наталья Виртуозова.\r\n\r\n«Третьим по счету заместителем Воробьева станет Юрий Олейников. Он будет курировать политические вопросы, вопросы взаимодействия с общественными организациями и муниципальным образованиями», — сообщила Виртуозова.\r\n\r\nЕго кандидатуру в ближайшие дни должна утвердить Мособлдума, заключила она.\r\n\r\nОлейников в 2005-2008 годах был заместителем руководителя ЦИК партии «Единая Россия» по региональной политике, с 5 февраля 2010 года — заместителем полномочн

In [11]:
def convert_markup_to_ner(item):
    text = item.text
    tokens = []
    offsets = []
    for match in re.finditer(r"\S+", text):
        tokens.append(match.group())
        offsets.append((match.start(), match.end()))

    labels = ["O"] * len(tokens)

    for span in item.spans:
        token_indices = [
            i for i, (tstart, tend) in enumerate(offsets) if not (tend <= span.start or tstart >= span.stop)
        ]
        if token_indices:
            labels[token_indices[0]] = "B-" + span.type
            for idx in token_indices[1:]:
                labels[idx] = "I-" + span.type

    return {"id": item.id, "tokens": tokens, "ner_tags": labels}

In [12]:
data_list = [convert_markup_to_ner(item) for item in dataset]
ner_dataset = Dataset.from_list(data_list)

In [13]:
unique_labels = set()
for example in ner_dataset:
    unique_labels.update(example["ner_tags"])

unique_labels = sorted(list(unique_labels))
label_to_id = {label: i for i, label in enumerate(unique_labels)}

In [14]:
label_to_id

{'B-GEOPOLIT': 0,
 'B-LOC': 1,
 'B-MEDIA': 2,
 'B-ORG': 3,
 'B-PER': 4,
 'I-GEOPOLIT': 5,
 'I-LOC': 6,
 'I-MEDIA': 7,
 'I-ORG': 8,
 'I-PER': 9,
 'O': 10}

In [15]:
def convert_labels(example):
    example["ner_tags"] = [label_to_id[label] for label in example["ner_tags"]]
    return example

In [16]:
ner_dataset = ner_dataset.map(convert_labels)

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

In [17]:
features = ner_dataset.features.copy()
features["ner_tags"] = Sequence(ClassLabel(names=unique_labels))
ner_dataset = ner_dataset.cast(features)

Casting the dataset:   0%|          | 0/999 [00:00<?, ? examples/s]

In [18]:
print(ner_dataset[0])

{'id': '611', 'tokens': ['Трудный', 'путь', 'к', 'солнцу:', 'согреет', 'ли', 'Россию', 'солнечная', 'энергетика', 'Перспективы', 'развития', 'солнечной', 'энергетики', 'в', 'России', 'остаются', 'весьма', 'неопределенными.', 'Правительство', 'в', 'конце', 'мая', 'приняло', 'пакет', 'документов,', 'призванных', 'стимулировать', 'развитие', 'в', 'РФ', 'возобновляемой', 'энергетики,', 'что', 'поможет', 'привлечь', 'в', 'эту', 'сферу', 'инвесторов.', 'Общая', 'мощность', 'солнечной', 'энергетики', 'в', 'России', 'к', '2020г.', 'может', 'вырасти', 'практически', 'в', '1000', 'раз,', 'однако', 'ее', 'доля', 'в', 'общем', 'энергобалансе', 'РФ', 'все', 'равно', 'останется', 'незначительной', 'и', 'несопоставимой', 'с', 'ситуацией', 'в', 'европейских', 'странах.', 'Эксперты', 'отмечают,', 'что', 'развитию', 'альтернативной', 'энергетики', 'в', 'РФ', 'мешает', 'непроработанность', 'технической', 'и', 'правовой', 'базы', 'и', 'недостаточное', 'производство', 'оборудования.', 'По', 'их', 'оценкам,

In [19]:
tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.74M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [20]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128,
        padding="max_length"
    )
    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        all_labels.append(label_ids)
    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

In [21]:
tokenized_dataset = ner_dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

In [22]:
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.2)

In [23]:
train_dataset = tokenized_dataset["train"]
test_dataset = tokenized_dataset["test"]

In [24]:
num_labels = len(unique_labels)
model = AutoModelForTokenClassification.from_pretrained("cointegrated/rubert-tiny2", num_labels=num_labels)

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
training_args = TrainingArguments(
    output_dir="models/ner_results",
    eval_strategy="steps",
    save_strategy="steps",
    save_steps=1000,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    max_steps=10000,
    logging_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    report_to="none",
    load_best_model_at_end=True,
    save_total_limit=2,
)

In [26]:
metric = evaluate.load("seqeval")

In [27]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
)


In [28]:
print("Метрики ДО дообучения:")
pre_training_results = trainer.evaluate()
print(tabulate.tabulate(
    pre_training_results.items(),
    headers=["Метрика", "Значение"],
    tablefmt="grid",
    floatfmt=".4f"
))

Метрики ДО дообучения:


+-----------------------------+------------+
| Метрика                     |   Значение |
+=============================+============+
| eval_loss                   |     2.3465 |
+-----------------------------+------------+
| eval_model_preparation_time |     0.0050 |
+-----------------------------+------------+
| eval_runtime                |     1.5734 |
+-----------------------------+------------+
| eval_samples_per_second     |   127.1160 |
+-----------------------------+------------+
| eval_steps_per_second       |     8.2630 |
+-----------------------------+------------+


In [29]:
trainer.train()

Step,Training Loss,Validation Loss,Model Preparation Time
100,1.036700,0.535016,0.005000
200,0.420200,0.305974,0.005000
300,0.257300,0.203146,0.005000
400,0.175800,0.159219,0.005000
500,0.131100,0.136838,0.005000
600,0.103600,0.118951,0.005000
700,0.084600,0.111450,0.005000
800,0.069900,0.105428,0.005000
900,0.059700,0.101234,0.005000
1000,0.051200,0.098704,0.005000


TrainOutput(global_step=10000, training_loss=0.02930684951543808, metrics={'train_runtime': 373.3634, 'train_samples_per_second': 428.537, 'train_steps_per_second': 26.784, 'total_flos': 282960319641600.0, 'train_loss': 0.02930684951543808, 'epoch': 200.0})

In [30]:
print("Метрики ПОСЛЕ дообучения:")
post_training_results = trainer.evaluate()
print(tabulate.tabulate(
    post_training_results.items(),
    headers=["Метрика", "Значение"],
    tablefmt="grid",
    floatfmt=".4f"
))

Метрики ПОСЛЕ дообучения:


+-----------------------------+------------+
| Метрика                     |   Значение |
+=============================+============+
| eval_loss                   |     0.0987 |
+-----------------------------+------------+
| eval_model_preparation_time |     0.0050 |
+-----------------------------+------------+
| eval_runtime                |     0.1321 |
+-----------------------------+------------+
| eval_samples_per_second     |  1514.4710 |
+-----------------------------+------------+
| eval_steps_per_second       |    98.4410 |
+-----------------------------+------------+
| epoch                       |   200.0000 |
+-----------------------------+------------+
